# Air Quality Forecasting — 24-Hour PM2.5 Prediction

**Multivariate time-series RNN benchmark with honest failure-mode analysis.**

This notebook trains six RNN architectures to forecast hourly PM2.5 24 hours ahead, benchmarks them against three baselines (naive, seasonal naive, 7-day same-hour mean), and explicitly diagnoses the regression-to-the-mean behaviour that point-estimate metrics hide.

## Contents
1. Setup & data loading
2. Exploratory data analysis
3. Feature engineering & leakage-safe split
4. Baselines (three of them, not just one)
5. RNN model architectures
6. Training loop
7. Results table
8. Horizon-wise performance
9. **Failure-mode analysis** — peak vs trough vs typical-day breakdown
10. Real vs Predicted — visual diagnostics
11. What I'd do next

## 1. Setup & data loading

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow.keras.backend as K

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow {tf.__version__}")

In [ ]:
# ----- Load -----
DATA_PATH = "Dataset_Air_Quality.csv"  # adjust if needed
df = pd.read_csv(DATA_PATH)

# Parse datetime (this dataset uses day-first format: '1/03/2013 0:00')
df['datetime'] = pd.to_datetime(df['datetime'], dayfirst=True, errors='coerce')
df = df.dropna(subset=['datetime']).sort_values('datetime').set_index('datetime')

print("Shape:", df.shape)
print("Date range:", df.index.min(), "→", df.index.max())
print()
print("First rows:")
df.head()

In [ ]:
# ----- Missing-value check -----
print("NaN counts per column:")
print(df.isna().sum())
print()
print(f"Total NaN cells: {df.isna().sum().sum()}")

In [ ]:
# ----- Forward-fill then back-fill -----
# ffill keeps last known value; bfill handles any leading NaNs
df = df.ffill().bfill()
print("NaN cells after fill:", df.isna().sum().sum())

## 2. Exploratory Data Analysis

Three things to check before modelling:
1. **Coverage** — are there months with sparse data?
2. **Target distribution** — heavy tails matter for choice of loss function
3. **Diurnal pattern** — does the network even need to learn the time-of-day effect, or do we feed it as a feature?

In [ ]:
target_col = 'PM2.5'

# A1) Coverage by month
rows_per_month = df.resample('ME').size()  # 'ME' is the modern alias for 'M' (avoids deprecation warning)
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.bar(rows_per_month.index, rows_per_month.values, width=20)
ax.set_title("Rows per month (data coverage)")
ax.set_ylabel("Rows")
ax.set_xlabel("Month")
plt.tight_layout()
plt.show()

In [ ]:
# A2) Target distribution with key percentiles
pm = df[target_col].astype(float)
p50, p95, p99 = np.percentile(pm, [50, 95, 99])

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(pm, bins=50)
ax.axvline(p50, color='tab:green', ls='--', label=f'p50 ≈ {p50:.0f}')
ax.axvline(p95, color='tab:orange', ls='--', label=f'p95 ≈ {p95:.0f}')
ax.axvline(p99, color='tab:red', ls='--', label=f'p99 ≈ {p99:.0f}')
ax.set_title(f"{target_col} distribution — heavy right tail")
ax.set_xlabel(target_col)
ax.legend()
plt.tight_layout()
plt.show()

print(f"Heavy tail: p99 ({p99:.0f}) is {p99/p50:.1f}x the median ({p50:.0f}).")
print("This matters — a model that predicts the conditional mean will systematically under-call peak days.")

In [ ]:
# A3) Diurnal seasonality (average PM2.5 by hour of day)
hod = df.groupby(df.index.hour)[target_col].mean()

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(hod.index, hod.values, marker='o')
ax.set_title(f"Average {target_col} by hour of day")
ax.set_xlabel("Hour")
ax.set_ylabel(target_col)
ax.set_xticks(range(0, 24, 2))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Feature Engineering & Leakage-Safe Split

Cyclic time encodings (`sin/cos` of hour and day-of-week) so the network doesn't have to learn that hour 23 and hour 0 are adjacent.

**Split strategy:**
- **Train:** all data ≤ 2015-12-31 23:00
- **Validation:** last 10% of training period (contiguous, time-ordered — no shuffling)
- **Test:** January 2016 (strict hold-out)
- **Bridging:** the last 504 training hours are prepended to the test set so the first Jan-1 forecast has full lookback context. Uses past data only, no leakage.

In [ ]:
# ----- Feature list -----
# Build candidate list, keep only columns that actually exist (PM10 isn't in this dataset)
candidate_features = [target_col, 'PM10', 'SO2', 'NO2', 'CO', 'O3',
                      'TEMP', 'PRES', 'DEWP', 'WSPM', 'RAIN']
feature_cols = [c for c in candidate_features if c in df.columns]
# Ensure target stays first (used by the make_windows function)
feature_cols = [target_col] + [c for c in feature_cols if c != target_col]
print("Pollutant + weather features:", feature_cols)

# Working frame
data = df[feature_cols].copy()

# ----- Cyclic time features -----
hr = data.index.hour.values
dw = data.index.dayofweek.values
data['sin_hour'] = np.sin(2 * np.pi * hr / 24)
data['cos_hour'] = np.cos(2 * np.pi * hr / 24)
data['sin_dow']  = np.sin(2 * np.pi * dw / 7)
data['cos_dow']  = np.cos(2 * np.pi * dw / 7)

# Final feature list (target first, time features last)
feature_cols = ([target_col]
                + [c for c in feature_cols if c != target_col]
                + ['sin_hour', 'cos_hour', 'sin_dow', 'cos_dow'])
print(f"Total features: {len(feature_cols)}")
print("Feature order (target first):", feature_cols)

In [ ]:
# ----- Time-ordered split -----
TRAIN_END = '2015-12-31 23:00:00'
TEST_START = '2016-01-01 00:00:00'
TEST_END   = '2016-01-31 23:59:59'

train_df = data.loc[:TRAIN_END].copy()
test_df  = data.loc[TEST_START:TEST_END].copy()

# Validation = last 10% of training (contiguous, no shuffle)
val_size = max(1, int(len(train_df) * 0.10))
val_df   = train_df.iloc[-val_size:].copy()
train_df = train_df.iloc[:-val_size].copy()

print(f"Train:      {train_df.shape}  ({train_df.index.min()} → {train_df.index.max()})")
print(f"Validation: {val_df.shape}  ({val_df.index.min()} → {val_df.index.max()})")
print(f"Test:       {test_df.shape}  ({test_df.index.min()} → {test_df.index.max()})")

In [ ]:
# ----- Visualise the split -----
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(data.index, data[target_col], lw=0.6, color='tab:blue', alpha=0.7)
ax.axvspan(train_df.index.min(), train_df.index.max(), color='tab:gray',   alpha=0.15, label='Train')
ax.axvspan(val_df.index.min(),   val_df.index.max(),   color='tab:orange', alpha=0.20, label='Validation')
ax.axvspan(test_df.index.min(),  test_df.index.max(),  color='tab:green',  alpha=0.25, label='Test (Jan-2016)')
ax.set_title(f"{target_col} over time — train / val / test split")
ax.set_ylabel(target_col)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
# ----- Scale (fit on TRAIN ONLY) -----
feat_scaler = MinMaxScaler().fit(train_df[feature_cols].values)
tgt_scaler  = MinMaxScaler().fit(train_df[[target_col]].values)

def scale_block(block):
    X = feat_scaler.transform(block[feature_cols].values)
    y = tgt_scaler.transform(block[[target_col]].values)
    return X, y

Xtr_s, ytr_s = scale_block(train_df)
Xva_s, yva_s = scale_block(val_df)
Xte_s, yte_s = scale_block(test_df)

print(f"Scaled: train {Xtr_s.shape}, val {Xva_s.shape}, test {Xte_s.shape}")

In [ ]:
# ----- Sliding windows (21-day lookback, 24-hour direct multi-step output) -----
LOOKBACK = 24 * 21  # 504 hours
HORIZON  = 24

def make_windows(feat, tgt, lookback=LOOKBACK, horizon=HORIZON):
    X, Y = [], []
    T = len(tgt)
    for t in range(lookback, T - horizon + 1):
        X.append(feat[t - lookback:t, :])
        Y.append(tgt[t:t + horizon, 0])
    return np.array(X), np.array(Y)

X_train, y_train = make_windows(Xtr_s, ytr_s)
X_val,   y_val   = make_windows(Xva_s, yva_s)

# Test windows: bridge the last LOOKBACK rows of train so the first Jan-1 forecast has context
bridge   = train_df.iloc[-LOOKBACK:].copy()
test_for_windows = pd.concat([bridge, test_df])
Xte_b_s, yte_b_s = scale_block(test_for_windows)
X_all, y_all     = make_windows(Xte_b_s, yte_b_s)

# Keep only windows whose forecast START falls within January 2016
all_starts = test_for_windows.index[LOOKBACK : LOOKBACK + len(y_all)]
mask = all_starts >= test_df.index.min()
X_test, y_test = X_all[mask], y_all[mask]
start_times    = all_starts[mask]

print(f"Windows -> train {X_train.shape}, val {X_val.shape}, test {X_test.shape}")
print(f"Test forecast windows span: {start_times.min()} → {start_times.max()}")

## 4. Baselines

The original notebook used only the weakest possible baseline (repeat-last). That makes the RNN look better than it should. Here we include three baselines so the comparison is honest:

1. **Naive (repeat-last):** repeat the last observed PM2.5 for all 24 future hours
2. **Seasonal naive (24h lag):** predict each hour `t+k` as the value from `t+k-24` — encodes the diurnal cycle for free
3. **7-day same-hour mean:** average of the last 7 days' value at the same hour-of-day — smooths out noise in the seasonal pattern

> 💡 **Counterintuitive result you may see:** for PM2.5 specifically, the simple repeat-last baseline often *beats* the seasonal baselines on this dataset. PM2.5 has very high hour-to-hour autocorrelation (this hour predicts next hour very well), and the diurnal cycle is weaker than that persistence signal. So "the air will be roughly what it is right now" turns out to be hard to beat. This raises the bar for the RNN: beating naive by 35% is **not** the right framing if naive is already the strongest of the simple baselines.

In [ ]:
def inverse_target(y_scaled):
    """Inverse-transform scaled target back to original PM2.5 units."""
    return tgt_scaler.inverse_transform(y_scaled.reshape(-1, 1)).reshape(y_scaled.shape)

def evaluate(y_true_s, y_pred_s, clip_nonneg=True):
    """Inverse-scale, optionally clip, then compute MAE / RMSE on original units."""
    y_true = inverse_target(y_true_s)
    y_pred = inverse_target(y_pred_s)
    if clip_nonneg:
        y_pred = np.clip(y_pred, 0, None)
    mae  = mean_absolute_error(y_true.ravel(), y_pred.ravel())
    rmse = np.sqrt(mean_squared_error(y_true.ravel(), y_pred.ravel()))
    return mae, rmse, y_true, y_pred

In [ ]:
# ----- Baseline 1: Naive repeat-last -----
def naive_repeat_last(X):
    last = X[:, -1, 0:1]                         # last target value of each window
    return np.repeat(last, HORIZON, axis=1)

# ----- Baseline 2: Seasonal naive (24h lag) -----
def seasonal_naive_24(X):
    # Last 24 hours of the lookback window, used as the next 24-hour forecast
    return X[:, -24:, 0]

# ----- Baseline 3: 7-day same-hour mean -----
def mean_7d_same_hour(X):
    """For each forecast step k, average target at hours (-24-k, -48-k, ..., -168-k) of the lookback."""
    n = X.shape[0]
    out = np.zeros((n, HORIZON))
    # Within the lookback (504 hours), the last 7*24=168 hours are the most recent week.
    # For forecast hour k (k=0..23), the "same hour" 1, 2, ..., 7 days ago is at lookback indices:
    #   -24+k, -48+k, ..., -168+k  (all relative to end of lookback)
    last_week = X[:, -168:, 0]                   # shape (n, 168)
    # last_week[:, j] is hour j of the last week (j=0 is 168h ago, j=167 is the most recent hour)
    # The "same hour 1 day ago" of forecast step k is last_week[:, 168-24+k]; 2 days ago is last_week[:, 168-48+k]; ...
    for k in range(HORIZON):
        same_hour_indices = [168 - 24*d + k for d in range(1, 8)]  # 7 prior same-hour-of-day points
        # Filter out negatives (shouldn't happen since 168 - 24*7 + k = k >= 0)
        out[:, k] = last_week[:, same_hour_indices].mean(axis=1)
    return out

In [ ]:
# ----- Run all three baselines on test -----
baseline_results = []
baseline_preds   = {}

for name, fn in [("Naive (repeat-last)",       naive_repeat_last),
                 ("Seasonal naive (24h lag)",  seasonal_naive_24),
                 ("7-day same-hour mean",      mean_7d_same_hour)]:
    yhat_s = fn(X_test)
    mae, rmse, y_true_inv, yhat_inv = evaluate(y_test, yhat_s)
    baseline_results.append((name, mae, rmse))
    baseline_preds[name] = yhat_inv
    print(f"{name:30s}  MAE = {mae:.2f}  RMSE = {rmse:.2f}")

# Cache true values once for later use
y_true_test_inv = y_true_inv

## 5. RNN Architectures

Six variants under identical training conditions. Each builder uses the modern Keras pattern (`Input` layer instead of `input_shape=` argument, which is deprecated).

A seventh variant — **GRU-128 with peak-aware loss** — is added to test whether asymmetric loss reduces the under-prediction issue on peak days.

In [ ]:
input_shape = (LOOKBACK, X_train.shape[2])

def build_lstm(units=128, dropout_rate=None, loss='mae'):
    m = Sequential([Input(shape=input_shape), LSTM(units)])
    if dropout_rate:
        m.add(Dropout(dropout_rate))
    m.add(Dense(HORIZON))
    m.compile(optimizer='adam', loss=loss)
    return m

def build_stacked_lstm(u1=128, u2=64, loss='mae'):
    m = Sequential([
        Input(shape=input_shape),
        LSTM(u1, return_sequences=True),
        LSTM(u2),
        Dense(HORIZON),
    ])
    m.compile(optimizer='adam', loss=loss)
    return m

def build_gru(units=128, loss='mae'):
    m = Sequential([
        Input(shape=input_shape),
        GRU(units),
        Dense(HORIZON),
    ])
    m.compile(optimizer='adam', loss=loss)
    return m

# Peak-aware loss: penalise under-prediction more than over-prediction.
# This directly targets the regression-to-mean failure mode.
def asymmetric_mae(under_weight=2.0):
    def loss_fn(y_true, y_pred):
        err = y_true - y_pred
        # When err > 0 we under-predicted; weight that more heavily.
        return K.mean(K.maximum(under_weight * err, -err))
    return loss_fn

def build_gru_peak_aware(units=128):
    m = Sequential([
        Input(shape=input_shape),
        GRU(units),
        Dense(HORIZON),
    ])
    m.compile(optimizer='adam', loss=asymmetric_mae(under_weight=2.0))
    return m

In [ ]:
experiments = [
    ("LSTM-128",                     build_lstm(128, None, 'mae')),
    ("StackedLSTM 128->64",          build_stacked_lstm(128, 64, 'mae')),
    ("GRU-128",                      build_gru(128, 'mae')),
    ("LSTM-128 + Dropout 0.1",       build_lstm(128, 0.1, 'mae')),
    ("LSTM-128 (MSE loss)",          build_lstm(128, None, 'mse')),
    ("GRU-128 (no dropout)",         build_gru(128, 'mae')),
    ("GRU-128 (peak-aware)",         build_gru_peak_aware(128)),
]

EPOCHS, BATCH = 50, 64  # 50 epochs is plenty given EarlyStopping; cuts notebook runtime
callbacks = [EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)]

## 6. Training Loop

Each model trains under identical conditions: same data, same callbacks, same batch size. EarlyStopping with `restore_best_weights=True` ensures we evaluate the best validation checkpoint, not the last.

In [ ]:
results   = []
pred_store = {}

for name, model in experiments:
    print(f"\n=== Training {name} ===")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH,
        shuffle=False,                 # time series; never shuffle
        callbacks=callbacks,
        verbose=0,                     # quieter output for a clean notebook
    )
    epochs_run = len(history.history['loss'])
    best_val   = min(history.history['val_loss'])

    yhat_s = model.predict(X_test, verbose=0)
    mae, rmse, _, yhat_inv = evaluate(y_test, yhat_s)
    results.append((name, mae, rmse, epochs_run, best_val))
    pred_store[name] = yhat_inv
    print(f"  epochs run: {epochs_run}, best val_loss: {best_val:.4f}, test MAE: {mae:.2f}, RMSE: {rmse:.2f}")

## 7. Results

All seven RNN variants plus three baselines, ranked by test MAE.

In [ ]:
# Combine RNN + baseline results
all_results = (
    [(n, mae, rmse) for (n, mae, rmse, *_) in results]
    + baseline_results
)
results_df = (pd.DataFrame(all_results, columns=['Model', 'MAE', 'RMSE'])
              .sort_values('MAE')
              .reset_index(drop=True))
results_df.index = results_df.index + 1
print("=== Test set results (ranked by MAE) ===\n")
print(results_df.to_string())

# Save for reproducibility
results_df.to_csv('results.csv', index=False)
print("\nSaved to results.csv")

best_name = results_df.iloc[0]['Model']
print(f"\nBest model: {best_name}")

## 8. Horizon-Wise Performance

Errors should grow with lead time for a direct multi-step model. The shape of that growth tells us where the forecast can be trusted operationally.

In [ ]:
yhat_best = pred_store.get(best_name) if best_name in pred_store else baseline_preds.get(best_name)

# Per-horizon MAE / RMSE for the best model
mae_h  = [mean_absolute_error(y_true_test_inv[:, k], yhat_best[:, k]) for k in range(HORIZON)]
rmse_h = [np.sqrt(mean_squared_error(y_true_test_inv[:, k], yhat_best[:, k])) for k in range(HORIZON)]

h_err = pd.DataFrame({'Horizon (hr)': np.arange(1, HORIZON + 1),
                      'MAE': mae_h,
                      'RMSE': rmse_h})
print(h_err.head(8))

In [ ]:
# Horizon-wise plot, comparing best model to the strongest baseline
strongest_baseline = min(baseline_results, key=lambda r: r[1])  # lowest MAE among baselines
sb_name = strongest_baseline[0]
sb_pred = baseline_preds[sb_name]
sb_mae_h = [mean_absolute_error(y_true_test_inv[:, k], sb_pred[:, k]) for k in range(HORIZON)]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, HORIZON+1), mae_h,    label=f'{best_name} MAE',           color='tab:blue')
ax.plot(range(1, HORIZON+1), rmse_h,   label=f'{best_name} RMSE',          color='tab:blue', ls='--')
ax.plot(range(1, HORIZON+1), sb_mae_h, label=f'{sb_name} MAE (baseline)',  color='tab:gray')
ax.set_title(f"Horizon-wise error — {best_name} vs strongest baseline")
ax.set_xlabel("Forecast step (hours ahead)")
ax.set_ylabel("Error")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nIf the gap between '{best_name}' and '{sb_name}' is small,")
print("the RNN is doing little real work beyond what a seasonal baseline already provides.")

## 9. Failure-Mode Analysis — *the most important section*

A single MAE number hides massive variation. We split the test set into three regimes based on the *first* hour of each forecast window:

- **Peak windows** — first hour ≥ p90 of training PM2.5 (high-pollution episodes)
- **Trough windows** — first hour ≤ p10 of training PM2.5 (clean air)
- **Typical windows** — everything in between

A well-calibrated forecaster should perform roughly evenly across all three. A regression-to-the-mean forecaster will look fine on typical days and fail on both extremes.

In [ ]:
# Define peak / trough thresholds from TRAINING distribution (not test — that would leak)
train_pm = train_df[target_col].values
peak_threshold   = np.percentile(train_pm, 90)
trough_threshold = np.percentile(train_pm, 10)

print(f"Training distribution thresholds:")
print(f"  Trough (p10): ≤ {trough_threshold:.1f} µg/m³")
print(f"  Peak  (p90): ≥ {peak_threshold:.1f} µg/m³")

# Categorise test windows by the first true forecast hour (i.e. PM2.5 at t+1)
first_hour_true = y_true_test_inv[:, 0]
peak_mask   = first_hour_true >= peak_threshold
trough_mask = first_hour_true <= trough_threshold
typical_mask = ~(peak_mask | trough_mask)

print(f"\nTest windows by regime:")
print(f"  Peak    (≥ {peak_threshold:.0f}): {peak_mask.sum():3d} windows")
print(f"  Typical:                          {typical_mask.sum():3d} windows")
print(f"  Trough  (≤ {trough_threshold:.0f}): {trough_mask.sum():3d} windows")

In [ ]:
# Compute per-regime MAE for the best model AND every baseline
def regime_metrics(y_true, y_pred):
    out = {}
    for label, mask in [('Peak', peak_mask),
                        ('Typical', typical_mask),
                        ('Trough', trough_mask),
                        ('Overall', np.ones(len(y_true), dtype=bool))]:
        if mask.sum() == 0:
            out[label] = np.nan
        else:
            out[label] = mean_absolute_error(y_true[mask].ravel(), y_pred[mask].ravel())
    return out

# Build a comparison table: every baseline + every RNN variant
all_preds_for_compare = {}
for name in baseline_preds:
    all_preds_for_compare[name] = baseline_preds[name]
for name in pred_store:
    all_preds_for_compare[name] = pred_store[name]

regime_rows = []
for name, preds in all_preds_for_compare.items():
    m = regime_metrics(y_true_test_inv, preds)
    regime_rows.append({
        'Model': name,
        'Peak MAE': m['Peak'],
        'Typical MAE': m['Typical'],
        'Trough MAE': m['Trough'],
        'Overall MAE': m['Overall'],
    })

regime_df = pd.DataFrame(regime_rows).sort_values('Overall MAE').reset_index(drop=True)
regime_df.index = regime_df.index + 1
print("=== MAE by regime ===\n")
print(regime_df.round(2).to_string())

regime_df.to_csv('regime_results.csv', index=False)

In [ ]:
# Visualise the failure mode
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(regime_df))
width = 0.22

ax.bar(x - 1.5*width, regime_df['Peak MAE'],    width, label='Peak (high-pollution)', color='tab:red')
ax.bar(x - 0.5*width, regime_df['Typical MAE'], width, label='Typical',               color='tab:gray')
ax.bar(x + 0.5*width, regime_df['Trough MAE'],  width, label='Trough (clean air)',    color='tab:green')
ax.bar(x + 1.5*width, regime_df['Overall MAE'], width, label='Overall',               color='tab:blue', alpha=0.6)

ax.set_xticks(x)
ax.set_xticklabels(regime_df['Model'], rotation=35, ha='right')
ax.set_ylabel('MAE (µg/m³)')
ax.set_title('Where does each model fail?  Peak / Typical / Trough breakdown')
ax.legend()
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("""
Read this chart carefully:
- A model dominated by red bars under-predicts peak days (regression to mean).
- A model with low typical MAE but high peak MAE has a 'safe but useless' problem
  — it scores well overall but fails on the days that operationally matter most.
- The peak-aware GRU should trade some typical-day MAE for better peak performance.
  If it doesn't, the architecture itself can't capture the peak signal at this horizon.
""")

## 10. Real vs Predicted — visual sanity checks

Aggregate metrics still hide a lot. Plotting individual forecast windows is the final honesty test.

In [ ]:
def plot_window(window_idx, models_to_show, title_suffix=""):
    """Plot true vs one or more model predictions for a single test window."""
    start = start_times[window_idx]
    idx   = pd.date_range(start, periods=HORIZON, freq='h')

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(idx, y_true_test_inv[window_idx], label='Actual', color='black', lw=2)
    for name in models_to_show:
        preds = all_preds_for_compare[name]
        ax.plot(idx, preds[window_idx], '--', label=name, lw=1.5)
    ax.set_title(f"Window {window_idx}: {start.strftime('%Y-%m-%d %H:%M')}{title_suffix}")
    ax.set_ylabel(target_col)
    ax.legend(loc='best')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

models_show = [best_name, 'Seasonal naive (24h lag)', '7-day same-hour mean']

# Pick representative windows from each regime
peak_idx    = np.where(peak_mask)[0][0]   if peak_mask.sum()    > 0 else 0
trough_idx  = np.where(trough_mask)[0][0] if trough_mask.sum()  > 0 else 0
typical_idx = np.where(typical_mask)[0][0] if typical_mask.sum() > 0 else 0

plot_window(peak_idx,    models_show, " — PEAK regime")
plot_window(typical_idx, models_show, " — TYPICAL regime")
plot_window(trough_idx,  models_show, " — TROUGH regime")

## 11. What I'd Do Next

The diagnostics above directly motivate the next iteration:

| Issue surfaced here | Roadmap response |
|---|---|
| Naive baseline was weak; seasonal naive may already be close to RNN | Always benchmark against seasonal + same-hour baselines from the start |
| Peak windows have far higher MAE than typical | Peak-aware loss (tested above), or hybrid model with a separate peak classifier |
| Single test month (Jan 2016) | Rolling-origin cross-validation across multiple months; report mean ± std |
| Point estimates only | Quantile forecasts (e.g. pinball loss) so we get uncertainty bands, not single numbers |
| Single station | Multi-station modelling — share signal across nearby monitors |
| Missing real-time drivers | Add traffic, fire/dust events, regional transport features |

The headline takeaway: **a forecasting project is only as honest as its failure-mode analysis. The MAE number alone would have suggested a deployable model. The peak/trough breakdown shows it isn't — and that's a finding, not a weakness.**

---

*Notebook by Phuong Viet Dang (Jackie) — [LinkedIn](https://www.linkedin.com/in/phuongviet1912/)*